In [10]:

#################################################################################################################################################################################################################################################################################
#CONEXION AL ORACLE

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM CMRAS10", con=connection2)

connection2.close()




In [11]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()



base2.to_sql(name='tmp_cmras10', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL cmras10 FALSO CON LO OBTENIDO DEL ORACLE

query="""
ALTER TABLE tmp_cmras10 
ALTER COLUMN redasiscod TYPE CHARACTER (2),
ALTER COLUMN redasisdes TYPE CHARACTER (60),
ALTER COLUMN redasismeddes TYPE CHARACTER (20),
ALTER COLUMN redasisgercod TYPE CHARACTER (30),
ALTER COLUMN redasisestregcod TYPE CHARACTER (1);


UPDATE cmras10 
SET 

redasiscod= t.redasiscod,
redasisdes= t.redasisdes,
redasismeddes= t.redasismeddes,
redasisgercod= t.redasisgercod,
redasisestregcod= t.redasisestregcod




FROM tmp_cmras10 t 
WHERE cmras10.redasiscod = t.redasiscod and cmras10.redasiscod IS NOT NULL and t.redasiscod IS NOT NULL ;

INSERT INTO cmras10 
(redasiscod, redasisdes, redasismeddes, redasisgercod, redasisestregcod) 

SELECT 
redasiscod, redasisdes, redasismeddes, redasisgercod, redasisestregcod

FROM tmp_cmras10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM cmras10 
    WHERE cmras10.redasiscod = tmp_cmras10.redasiscod and cmras10.redasiscod IS NOT NULL and tmp_cmras10.redasiscod IS NOT NULL
);


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_cmras10;
DELETE FROM cmras10 WHERE redasiscod ='';
"""


c2= text(query2)
connection3.execute(c2)
base2 = pd.read_sql_query(f"SELECT * FROM cmras10", con=connection3)

connection3.close()


In [12]:
#################################################################################################################################################################################################################################################################################


#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="INDICADORES_ESSALUD"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine4 = create_engine(cadena4)
connection4 = engine4.connect()


base2.to_sql(name='tmp_cmras10', con=engine4, if_exists='replace', index=False)



#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS

query_count_before = "SELECT COUNT(*) FROM dim_red"
result_count_before = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla dim_red antes de la inserción: {result_count_before}")


query="""

ALTER TABLE tmp_cmras10 
ALTER COLUMN redasiscod TYPE CHARACTER (2),
ALTER COLUMN redasisdes TYPE CHARACTER (60),
ALTER COLUMN redasismeddes TYPE CHARACTER (20),
ALTER COLUMN redasisgercod TYPE CHARACTER (30),
ALTER COLUMN redasisestregcod TYPE CHARACTER (1);


INSERT INTO dim_red (cod_red, des_red,dco_red) 
SELECT redasiscod, redasisdes, redasismeddes  

FROM tmp_cmras10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM dim_red 
    WHERE dim_red.cod_red = tmp_cmras10.redasiscod
);

DROP TABLE tmp_cmras10;

SELECT COUNT(*) FROM dim_red;
"""

c1 = text(query)
cursor = connection4.execute(c1)
inserted_rows = cursor.fetchone()[0]
print(f"Se insertaron {inserted_rows} filas en la tabla dim_red.")
connection4.close()

Cantidad de filas en la tabla dim_red antes de la inserción: 34
Se insertaron 34 filas en la tabla dim_red.
